# AMD Ryzen™ AI Software — NPU Inference on Linux Walkthrough
**2026 DSP & AI Summit · Thomas Zerbs**

---

## End-to-End Flow

| Step | What | Tool |
|:----:|------|------|
| 1 | Load pre-trained PyTorch model | `torchvision` |
| 2 | Export to ONNX | `torch.onnx.export()` |
| 3 | Quantize | AMD Quark (INT8) · VAIML auto-cast (BF16) |
| 4 | Compile, cache & infer | ONNX Runtime + Vitis AI EP |
| 5 | Analyze | AI Analyzer · `xrt-smi` |

| Tutorial | Model | Focus |
|----------|-------|-------|
| **A** | ResNet50 INT8 | Quark PTQ → XINT8 ONNX → VitisAI EP |
| **B** | ResNet50 BF16 | FP32 ONNX → VAIML auto-cast → VitisAI EP |
| **C** | YOLOv8m | AI Analyzer, `xrt-smi`, benchmarking |

> **System**: HP Z2 Mini PC (Strix Halo APU) · Ubuntu 24.04.3 · Kernel 6.17 · Ryzen AI SW 1.7.1

In [5]:
import os, sys
from pathlib import Path

REPO     = Path('/scratch/thozerbs/git/RyzenAI-SW')
INT8_DIR = REPO / 'CNN-examples/getting_started_resnet/int8'
BF16_DIR = REPO / 'CNN-examples/getting_started_resnet/bf16'
YOLO_DIR = REPO / 'CNN-examples/object_detection/yolov8m'
BENCH_DIR = REPO / 'onnx-benchmark'

os.chdir(INT8_DIR)
print(f'Working directory: {os.getcwd()}')

Working directory: /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/int8


In [6]:
# Verify NPU is detected — always run this first
!xrt-smi examine

System Configuration
  OS Name              : Linux
  Release              : 6.17.0-19-generic
  Machine              : x86_64
  CPU Cores            : 32
  Memory               : 96323 MB
  Distribution         : Ubuntu 24.04.3 LTS
  GLIBC                : 2.39
  Model                : HP Z2 Mini G1a Workstation Desktop PC
  BIOS Vendor          : HP
  BIOS Version         : X53 Ver. 01.02.03
  Processor            : AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S

XRT
  Version              : 2.21.75
  Branch               : HEAD
  Hash                 : 4eb1f4392a012b4e6eca759762389c612537f7c7
  Hash Date            : 2026-03-09 20:30:37
  amdxdna Version      : 2.21.260102.53.release_20260309, 6f881ad230142b707ca8ce5b33fca426a926c551
  virtio-pci Version   : 6.17.0-19-generic
  NPU Firmware Version : 1.1.2.65

Device(s) Present
|BDF             |Name            |
|----------------|----------------|
|[0000:c5:00.1]  |NPU Strix Halo  |




In [3]:
!aianalyzer --version

1.7.0.dev20260130181427+g301504b8


---
## Tutorial A — ResNet50 INT8
**Prepare Data · Load Model · Quantize · Deploy**  
`PyTorch` · `AMD Quark` · `XINT8` · `Vitis AI EP` · `ONNX Runtime`

### A1 — Load Data, Model & Export to ONNX

`prepare_model_data.py` automates three steps: download CIFAR-10, load pre-trained ResNet50, and export to ONNX.

**Key APIs** (`prepare_model_data.py`):

```python
from torchvision.models import resnet50, ResNet50_Weights

# 1. Load ResNet50 with pre-trained ImageNet weights
weights = ResNet50_Weights.DEFAULT
resnet = resnet50(weights=weights)

# Replace FC head: ImageNet has 1,000 classes, CIFAR-10 has 10
# Keep the full backbone, swap only the head: 2048 → 64 → 10
resnet.fc = nn.Sequential(nn.Linear(2048, 64), nn.ReLU(inplace=True), nn.Linear(64, 10))
```

```python
# 2. Export to ONNX
dummy_inputs = torch.randn(1, 3, 32, 32)       # CIFAR-10 input size
torch.onnx.export(
    model, dummy_inputs,
    'models/resnet_trained_for_cifar10.onnx',
    opset_version=17,                           # VitisAI EP / VAIML requires opset 17
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
```

> **ONNX opset 17** is required by VitisAI EP / VAIML. If your model uses a different opset version, convert it using the ONNX Version Converter.

**Output**: `models/resnet_trained_for_cifar10.onnx` — FP32 ONNX, ready for Quark quantization (INT8) or VAIML compilation (BF16).

In [4]:
# Download CIFAR-10, load trained checkpoint, export to ONNX
!python prepare_model_data.py

### A2 — Quantize with AMD Quark (INT8)

`resnet_quantize.py` — post-training quantization (PTQ). Quark converts the FP32 ONNX model to INT8 ONNX.

**Key APIs** (`resnet_quantize.py`):

```python
from quark.onnx.quantization.config import Config, get_default_config
from quark.onnx import ModelQuantizer

# 1. Pick a quantization config (XINT8 | A8W8 | A16W8 | BF16 | BFP16)
quant_config = get_default_config('XINT8')        # Symmetric INT8, power-of-two scales
config = Config(global_quant_config=quant_config)

# 2. Run post-training quantization with calibration data
quantizer = ModelQuantizer(config)
quantizer.quantize_model(
    input_model_path  = 'models/resnet_trained_for_cifar10.onnx',   # FP32 ONNX
    output_model_path = 'models/resnet_quantized.onnx',            # INT8 ONNX
    resnet_calibration_reader(calibration_dataset_path)             # calibration data reader
)
```

The `CalibrationDataReader` feeds representative CIFAR-10 images to calibrate activation ranges.

| | No calibration | With calibration | + AdaRound |
|---|---|---|---|
| **L2 loss** | 30.26 | 9.78 | **1.43** |

**Output**: `models/resnet_quantized.onnx` — standard ONNX, ready for VitisAI EP.

In [7]:
# Run PTQ — Quark converts FP32 ONNX → INT8 ONNX (~1-2 minutes)
!python resnet_quantize.py


[QUARK-INFO]: Checking custom ops library ...

[QUARK-INFO]: The CPU version of custom ops library already exists.

[QUARK-INFO]: Checked custom ops library.
The configuration for quantization is Config(global_quant_config=QuantizationConfig(calibrate_method=<PowerOfTwoMethod.MinMSE: 1>, quant_format=<QuantFormat.QDQ: 1>, activation_type=<QuantType.QUInt8: 1>, weight_type=<QuantType.QInt8: 0>, input_nodes=[], output_nodes=[], op_types_to_quantize=[], nodes_to_quantize=[], extra_op_types_to_quantize=[], nodes_to_exclude=[], subgraphs_to_exclude=[], specific_tensor_precision=False, execution_providers=['CPUExecutionProvider'], per_channel=False, reduce_range=False, optimize_model=True, use_dynamic_quant=False, use_external_data_format=False, convert_fp16_to_fp32=False, convert_nchw_to_nhwc=False, include_sq=False, include_rotation=False, include_cle=True, include_auto_mp=False, include_fast_ft=False, enable_npu_cnn=True, enable_npu_transformer=False, debug_mode=False, crypto_mode=False,

### A3 — Inference: CPU Baseline → NPU

`predict.py` creates an ONNX Runtime `InferenceSession` and runs predictions on 10 CIFAR-10 test images.  
**Best practice**: run CPU first to verify the model is correct, then switch to NPU.

**Key APIs** (`predict.py`):

```python
import onnxruntime as ort

# --- CPU (default, no flag) ---
providers        = ['CPUExecutionProvider']
provider_options = [{}]

# --- NPU via Vitis AI EP (--ep npu) ---
providers = ['VitisAIExecutionProvider']
provider_options = [{
    'cache_dir': str(cache_dir),       # compiled NPU binary saved here
    'cache_key': 'modelcachekey',      # change to force recompile when model changes
    'enable_cache_file_io_in_mem': '0',
}]

# Create ONNX RT session
session_options = ort.SessionOptions()
session_options.log_severity_level = 1  # 0=Verbose 1=Info 2=Warning 3=Error 4=Fatal

session = ort.InferenceSession(
    model.SerializeToString(),
    sess_options=session_options,
    providers=providers,
    provider_options=provider_options
)

# Run inference — identical API regardless of CPU or NPU
outputs = session.run(None, {'input': input_data})
predicted_class = np.argmax(outputs[0])
```

**Provider options for INT8 on STX/KRK**:
- `cache_dir` & `cache_key` — compiled binary cache location and unique model key
- `opt_level` — compiler optimization level (0, 1, 2, 3, 65536), INT8 only
- `enable_cache_file_io_in_mem` — keep compiled model in memory (1) or save to disk (0)
- If compiling INT8 on **PHX/HPT**: `xclbin` & `target` must be set

> **First NPU run** compiles the INT8 model (~60 s) and caches to `./modelcachekey/`. Subsequent runs load from cache in under 1 second. Predictions should match CPU output.

In [6]:
# Step 1: Run on CPU — verify model correctness (no NPU driver needed)
!python predict.py

2026-04-13 16:00:14.818541937 [I:onnxruntime:, inference_session.cc:606 TraceSessionOptions] Session Options {  execution_mode:0 execution_order:DEFAULT enable_profiling:0 optimized_model_filepath:"" enable_mem_pattern:1 enable_mem_reuse:1 enable_cpu_mem_arena:1 profile_file_prefix:onnxruntime_profile_ session_logid: session_log_severity_level:1 session_log_verbosity_level:0 max_num_graph_transformation_steps:10 graph_optimization_level:4 intra_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } inter_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } use_per_session_threads:1 thread_pool_allow_spinning:1 use_deterministic_compute:0 ep_selection_policy:0 config_options: {  } }
2026-04-13 16:00:14.818579638 [I:onnxruntime:, inference_session.cc:414 operator(

In [7]:
# Step 2: Run on NPU with Vitis AI EP
# First run: compiles INT8 model (~60 s), cached to ./modelcachekey/
# Subsequent runs: loads from cache (<1 s)
!python predict.py --ep npu

2026-04-13 16:00:20.119656340 [I:onnxruntime:, inference_session.cc:606 TraceSessionOptions] Session Options {  execution_mode:0 execution_order:DEFAULT enable_profiling:0 optimized_model_filepath:"" enable_mem_pattern:1 enable_mem_reuse:1 enable_cpu_mem_arena:1 profile_file_prefix:onnxruntime_profile_ session_logid: session_log_severity_level:1 session_log_verbosity_level:0 max_num_graph_transformation_steps:10 graph_optimization_level:4 intra_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } inter_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } use_per_session_threads:1 thread_pool_allow_spinning:1 use_deterministic_compute:0 ep_selection_policy:0 config_options: {  } }
2026-04-13 16:00:20.119691366 [I:onnxruntime:, inference_session.cc:414 operator(

---
## Tutorial B — ResNet50 BF16
**Vitis AI Compiler Auto-Casting**  
`VAIML` · `BF16`

| Path | Tool | Quantization | Calibration data? |
|------|------|-------------|-------------------|
| **INT8** | AMD Quark | Full INT8 quantization **before** compilation | Yes — required |
| **BF16** | Vitis AI Compiler (VAIML) | Auto-casts FP32 → BF16 **during** compilation | No — not needed |

> *Quark does support BF16, however the VAIML automatic conversion is the integrated Ryzen AI path: BF16 lowering happens together with NPU partitioning in ONNX Runtime + Vitis AI EP, and without calibration data.*

Uses the same FP32 ONNX model from Tutorial A.

In [4]:
os.chdir(BF16_DIR)
print(f'Working directory: {os.getcwd()}')

Working directory: /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/getting_started_resnet/bf16


### B1 — Compile FP32 Model to BF16

`compile.py` — VAIML auto-casts FP32 weights and activations to BF16, saves NPU binary artifacts to cache.  
Compilation is triggered by creating an `InferenceSession` — no separate compile step needed.

**`vitisai_config.json`** — the key field is `enable_f32_to_bf16_conversion: true`:
```json
{
    "passes": [
        { "name": "init", "plugin": "vaip-pass_init" },
        { "name": "vaiml_partition", "plugin": "vaip-pass_vaiml_partition",
          "vaiml_config": { "enable_f32_to_bf16_conversion": true } }
    ],
    "target": "VAIML"
}
```

**Key APIs** (`compile.py`):

```python
provider_options_dict = {
    'config_file':                config_file,   # points to vitisai_config.json
    'cache_dir':                  cache_dir,     # NPU binary artifacts saved here
    'cache_key':                  cache_key,     # change when model changes to recompile
    'enable_cache_file_io_in_mem': 0,
    'ai_analyzer_visualization':  True,          # graph partition + op fusion (JSON)
    'ai_analyzer_profiling':      True,          # per-operator timing (JSON)
}

# Compilation triggered by creating the InferenceSession
session = onnxruntime.InferenceSession(
    onnx_model,
    providers=['VitisAIExecutionProvider'],
    provider_options=[provider_options_dict]
)
```

**Output**: NPU-optimized binary artifacts in `my_cache_dir/`

In [9]:
# Compile FP32 ONNX → BF16 NPU binary (~60 seconds)
# Uses the same FP32 ONNX model exported in Tutorial A
!python compile.py --model ../int8/models/resnet_trained_for_cifar10.onnx

Creating ORT inference session for model ../int8/models/resnet_trained_for_cifar10.onnx
2026-04-13 16:01:04.283039274 [I:onnxruntime:, inference_session.cc:606 TraceSessionOptions] Session Options {  execution_mode:0 execution_order:DEFAULT enable_profiling:0 optimized_model_filepath:"" enable_mem_pattern:1 enable_mem_reuse:1 enable_cpu_mem_arena:1 profile_file_prefix:onnxruntime_profile_ session_logid: session_log_severity_level:1 session_log_verbosity_level:0 max_num_graph_transformation_steps:10 graph_optimization_level:4 intra_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } inter_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } use_per_session_threads:1 thread_pool_allow_spinning:1 use_deterministic_compute:0 ep_selection_policy:0 config_options: 

### B2 — BF16 Inference

`predict.py` (BF16) loads compiled artifacts from cache and runs inference on 1000 CIFAR-10 images.

**Key APIs** (`predict.py` BF16 path):

```python
# BF16 provider options — config_file is REQUIRED (INT8 does not need it)
provider_options_dict = {
    'config_file':                config_file,  # required for BF16 / VAIML path
    'cache_dir':                  cache_dir,    # loads compiled NPU binary from here
    'cache_key':                  cache_key,    # must match key used during compile
    'enable_cache_file_io_in_mem': 0,
    'ai_analyzer_visualization':  True,
    'ai_analyzer_profiling':      True,
}

session_options.enable_profiling = True          # ORT profiler for CPU layers (AI Analyzer)

session = onnxruntime.InferenceSession(
    onnx_model_path,
    sess_options=session_options,
    providers=['VitisAIExecutionProvider'],
    provider_options=[provider_options_dict]
)

outputs = session.run(None, {input_name: input_data})
session.end_profiling()   # flush ORT profiler JSON for AI Analyzer
```

> `session.end_profiling()` flushes the ORT native profiler data to a JSON file — this captures CPU-side layer timings for the AI Analyzer Performance panel.

In [10]:
# Run BF16 inference on NPU — loads from cache (<1 s)
!python predict.py --ep npu

execution started on NPU
2026-04-13 16:01:42.014343183 [I:onnxruntime:, inference_session.cc:606 TraceSessionOptions] Session Options {  execution_mode:0 execution_order:DEFAULT enable_profiling:1 optimized_model_filepath:"" enable_mem_pattern:1 enable_mem_reuse:1 enable_cpu_mem_arena:1 profile_file_prefix:onnxruntime_profile_ session_logid: session_log_severity_level:1 session_log_verbosity_level:0 max_num_graph_transformation_steps:10 graph_optimization_level:4 intra_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } inter_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } use_per_session_threads:1 thread_pool_allow_spinning:1 use_deterministic_compute:0 ep_selection_policy:0 config_options: {  } }
2026-04-13 16:01:42.014389300 [I:onnxruntime:, inference

---
## Tutorial C — YOLOv8m
**Analysis · NPU Management · Performance**  
`AI Analyzer` · `xrt-smi` · `Benchmark`

Uses YOLOv8m BF16 (pre-compiled) to demonstrate:
1. **AI Analyzer** — visualize NPU/CPU partitioning and per-operator profiling
2. **`xrt-smi`** — NPU management, performance mode, partition monitoring
3. **Benchmarking** — latency and FPS measurement

In [11]:
os.chdir(YOLO_DIR)
print(f'Working directory: {os.getcwd()}')

Working directory: /scratch/thozerbs/git/RyzenAI-SW/CNN-examples/object_detection/yolov8m


### C1 — AI Analyzer

**Purpose**: Visualize and analyze model compilation and inference on Vitis AI — understand NPU/CPU partitioning and identify performance bottlenecks.

Two provider option flags enable AI Analyzer data collection:

| Flag | Generates | Powers |
|------|-----------|--------|
| `ai_analyzer_visualization: True` | JSON: graph partition, op fusion | **Partitioning** + **NPU Insights** panels |
| `ai_analyzer_profiling: True` | JSON: per-operator timing (NPU) | **Performance** panel |
| `session_options.enable_profiling = True` (ORT) | JSON: CPU-side layer timings | Full CPU+NPU pipeline in Performance panel |

**Key APIs** (`run_inference.py`):

```python
provider_options = [{
    'config_file':               'vaiml_config.json',
    'cache_dir':                 str(Path('.').resolve()),
    'cache_key':                 'modelcachekey',
    'ai_analyzer_visualization': True,    # graph partition + op fusion artifacts
    'ai_analyzer_profiling':     True,    # per-operator timing artifacts
}]

session_options.enable_profiling = True    # ORT profiler for CPU layers

session = ort.InferenceSession(onnx_path,
    sess_options=session_options,
    providers=['VitisAIExecutionProvider'],
    provider_options=provider_options)

outputs = session.run(None, {input_name: input_img})
session.end_profiling()   # flush ORT profiler JSON
```

> **Note**: AI Analyzer flags must be set when **compiling** (first run) *and* when running **inference**.

In [12]:
# Run YOLOv8m BF16 on NPU — generates AI Analyzer JSON artifacts
!python run_inference.py \
    --model_input models/yolov8m_BF16.onnx \
    --input_image test_image.jpg \
    --output_image test_output.jpg \
    --device npu-bf16

Running BF16 Model on NPU
2026-04-13 16:02:17.682112420 [I:onnxruntime:, inference_session.cc:606 TraceSessionOptions] Session Options {  execution_mode:0 execution_order:DEFAULT enable_profiling:0 optimized_model_filepath:"" enable_mem_pattern:1 enable_mem_reuse:1 enable_cpu_mem_arena:1 profile_file_prefix:onnxruntime_profile_ session_logid: session_log_severity_level:1 session_log_verbosity_level:0 max_num_graph_transformation_steps:10 graph_optimization_level:4 intra_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } inter_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } use_per_session_threads:1 thread_pool_allow_spinning:1 use_deterministic_compute:0 ep_selection_policy:0 config_options: {  } }
2026-04-13 16:02:17.682135805 [I:onnxruntime:, inferenc

In [13]:
# Launch AI Analyzer — opens in browser at http://localhost:8000
# Interrupt kernel (Ctrl+C) to stop the server when done
!aianalyzer .

2026-04-13 16:02:29,961     INFO [client_id=n/a] 126884806391488 json_aie_record_timer.py:318 /model.22/Split_1_Duplicated#1 layer not found
2026-04-13 16:02:29,961     INFO [client_id=n/a] 126884806391488 json_aie_record_timer.py:318 /model.22/Split_1_Duplicated#0 layer not found
2026-04-13 16:02:30,063     INFO [client_id=n/a] 126884806391488 fxml_schedule_view.py:2921 Buffer layer /model.22/Sigmoid (202)
2026-04-13 16:02:32,309     INFO [client_id=n/a] 126884758726336 server.py:35 AI Analyzer 1.7.0.dev20260130181427+g301504b8 serving on http://localhost:8000/dashboard?token=OMaimaAZxFIMx5Tdrr89VdMIbZigkzs9c8aTbZyHpFs (Press CTRL+C to quit)
2026-04-13 16:02:35,309     INFO [client_id=n/a] 126884806391488 browser.py:30 Opening http://localhost:8000/dashboard?token=OMaimaAZxFIMx5Tdrr89VdMIbZigkzs9c8aTbZyHpFs in the default web browser...
^C


### C2 — NPU Management with `xrt-smi`

`xrt-smi` is the NPU management utility — analogous to `nvidia-smi` for NVIDIA GPUs.

| Command | Purpose |
|---------|----------|
| `xrt-smi examine` | System info: OS, XRT version, NPU driver/firmware, device BDF and name |
| `xrt-smi examine --report platform` | Performance mode and estimated power draw (Watts) |
| `xrt-smi examine --report aie-partitions` | Run **while a model is active** — NPU partition and column occupancy |
| `xrt-smi validate --run all` | Standalone sanity tests: no-op latency, DMA throughput, gemm INT8 TOPS |
| `xrt-smi configure --pmode performance` | Set performance mode (powersaver \| balanced \| performance \| turbo) |

In [15]:
# System info: OS, XRT version, NPU driver, device BDF and name
!xrt-smi examine

System Configuration
  OS Name              : Linux
  Release              : 6.17.0-19-generic
  Machine              : x86_64
  CPU Cores            : 32
  Memory               : 96323 MB
  Distribution         : Ubuntu 24.04.3 LTS
  GLIBC                : 2.39
  Model                : HP Z2 Mini G1a Workstation Desktop PC
  BIOS Vendor          : HP
  BIOS Version         : X53 Ver. 01.02.03
  Processor            : AMD RYZEN AI MAX+ PRO 395 w/ Radeon 8060S

XRT
  Version              : 2.21.75
  Branch               : HEAD
  Hash                 : 4eb1f4392a012b4e6eca759762389c612537f7c7
  Hash Date            : 2026-03-09 20:30:37
  amdxdna Version      : 2.21.260102.53.release_20260309, 6f881ad230142b707ca8ce5b33fca426a926c551
  virtio-pci Version   : 6.17.0-19-generic
  NPU Firmware Version : 1.1.2.65

Device(s) Present
|BDF             |Name            |
|----------------|----------------|
|[0000:c5:00.1]  |NPU Strix Halo  |




In [16]:
# Performance mode and estimated power draw (Watts)
!xrt-smi examine --report platform


--------------------------------
[0000:c5:00.1] : NPU Strix Halo
--------------------------------
Platform
  Name                   : NPU Strix Halo 
  Power Mode             : Default 
  Total Columns          : 8 

Estimated Power          : N/A



In [17]:
# NPU partition details and column occupancy
# Best run in a second terminal WHILE a model is actively running on the NPU
!xrt-smi examine --report aie-partitions


--------------------------------
[0000:c5:00.1] : NPU Strix Halo
--------------------------------
AIE Partitions
  No hardware contexts running on device



In [19]:
# Standalone NPU sanity tests: no-op latency, DMA throughput, gemm INT8 TOPS
!xrt-smi validate --run all --verbose

Verbose: Enabling Verbosity
Validate Device           : [0000:c5:00.1]
    Platform              : NPU Strix Halo
    Power Mode            : Default
-------------------------------------------------------------------------------
Test 1 [0000:c5:00.1]     : gemm 
    Description           : Measure the TOPS value of GEMM INT8 operations
[                    ]: Running Test... < 0s >
[<->                 ]: Running Test... < 0s >
[ <->                ]: Running Test... < 0s >
[  <->               ]: Running Test... < 0s >
[   <->              ]: Running Test... < 0s >
[    <->             ]: Running Test... < 0s >
[     <->            ]: Running Test... < 0s >
[      <->           ]: Running Test... < 0s >
[       <->          ]: Running Test... < 0s >
[        <->         ]: Running Test... < 0s >
[         <->        ]: Running Test... < 1s >
[          <->       ]: Running Test... < 1s >
[           <->      ]: Running Test... < 1s >
[            <->     ]: Running Test... < 1s >
[  

In [20]:
# Set NPU to performance mode before benchmarking
!xrt-smi configure --pmode performance

XRT build version: 2.21.75
Build hash: 4eb1f4392a012b4e6eca759762389c612537f7c7
Build date: 2026-03-09 20:30:37
Git branch: HEAD
PID: 3025090
UID: 50076701
[Mon Apr 13 23:04:27 2026 GMT]
HOST: xsjnucstrhalo03
EXE: /opt/xilinx/xrt/bin/unwrapped/xrt-smi
[xrt-smi] ERROR: DRM_IOCTL_AMDXDNA_SET_STATE IOCTL failed (err=-13): Permission denied


### C3 — Benchmarking

Two benchmarking approaches:

#### Built-in: `--benchmark` flag in `run_inference.py`
Measures end-to-end app performance (pre/post-processing included). Model-agnostic — works for any model by swapping session + model path.

```python
def benchmark(session, input_name, input_img, num_inference=100):
    # Warmup — 10 runs (driver/JIT settling)
    for _ in range(10):
        session.run(None, {input_name: input_img})

    # Timed inference loop
    time_list = []
    for _ in range(num_inference):
        start = time.time()
        output = session.run(None, {input_name: input_img})
        time_list.append(time.time() - start)

    avg = sum(time_list) / num_inference
    print(f'Avg time: {avg:.3f} s  |  {1/avg:.1f} FPS')
```

#### Standalone: `performance_benchmark.py` (onnx-benchmark tool)
Isolates pure inference with standardized reporting, power analysis, and GUI support.

```bash
python performance_benchmark.py \
    --model_path models/resnet_quantized.onnx \
    --config $RYZEN_AI_.../vaip_config.json \
    --execution_provider VitisAIEP \
    --autoquant 0 --renew 0 --num 100 --timelimit 10
```

| YOLOv8m | Avg latency | FPS |
|---------|------------|-----|
| Float32 (CPU) | 0.486 s | 2.1 |
| BF16 (NPU) | 0.075 s | 13.4 |
| XINT8 (NPU) | 0.021 s | 48.2 |

In [23]:
# Built-in benchmark: YOLOv8m BF16 on NPU (10 warmup + 100 timed runs)
!xrt-smi configure --pmode performance
!python run_inference.py \
    --model_input models/yolov8m_BF16.onnx \
    --input_image test_image.jpg \
    --output_image test_output.jpg \
    --device npu-bf16 \
    --benchmark

XRT build version: 2.21.75
Build hash: 4eb1f4392a012b4e6eca759762389c612537f7c7
Build date: 2026-03-09 20:30:37
Git branch: HEAD
PID: 3032589
UID: 50076701
[Mon Apr 13 23:27:42 2026 GMT]
HOST: xsjnucstrhalo03
EXE: /opt/xilinx/xrt/bin/unwrapped/xrt-smi
[xrt-smi] ERROR: DRM_IOCTL_AMDXDNA_SET_STATE IOCTL failed (err=-13): Permission denied
Running BF16 Model on NPU
2026-04-13 16:27:43.003135048 [I:onnxruntime:, inference_session.cc:606 TraceSessionOptions] Session Options {  execution_mode:0 execution_order:DEFAULT enable_profiling:0 optimized_model_filepath:"" enable_mem_pattern:1 enable_mem_reuse:1 enable_cpu_mem_arena:1 profile_file_prefix:onnxruntime_profile_ session_logid: session_log_severity_level:1 session_log_verbosity_level:0 max_num_graph_transformation_steps:10 graph_optimization_level:4 intra_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } inter_op_param:Ort

In [22]:
# Standalone onnx-benchmark tool: ResNet50 INT8 on NPU
# --autoquant 0 = model already quantized, skip Quark
# --renew 0     = reuse compile cache from Tutorial A
!python {BENCH_DIR}/performance_benchmark.py \
    --model_path {INT8_DIR}/models/resnet_quantized.onnx \
    --execution_provider VitisAIEP \
    --autoquant 0 --renew 0 --num 100 --timelimit 10


[QUARK-INFO]: Checking custom ops library ...

[QUARK-INFO]: The CPU version of custom ops library already exists.

[QUARK-INFO]: Checked custom ops library.
Timestamp = 0413_1606
Setting environment for STX
Set XLNX_VART_FIRMWARE=/scratch/thozerbs/ryzen_ai/venv/lib/python3.12/site-packages/flexml/flexml_extras/data/ryzen-ai/stx/unified-2x4x4.xclbin
------------------------------------------------------------------------------
Preliminary environment check
PACKAGE                             STATUS
Python version 3.12                 OK
onnxruntime-vitisai version: 1.24.1 OK
voe version: 1.7.1                  OK
Traceback (most recent call last):
  File "/scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark/performance_benchmark.py", line 496, in <module>
    bench.run()
  File "/scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark/performance_benchmark.py", line 84, in run
    check_args(self.args, self.defaults)
  File "/scratch/thozerbs/git/RyzenAI-SW/onnx-benchmark/utilities.py", line 626, i

In [ ]:
# CPU baseline for comparison
!python {BENCH_DIR}/performance_benchmark.py \
    --model_path {INT8_DIR}/models/resnet_quantized.onnx \
    --execution_provider CPU --num 100